<a href="https://www.arduino.cc/"><img src="https://raw.githubusercontent.com/sandeepmistry/aimldevfest-workshop-2019/master/images/Arduino_logo_R_highquality.png" width=200/></a>
# Tiny ML on Arduino
## Gesture recognition tutorial
 * Sandeep Mistry - Arduino
 * Don Coleman - Chariot Solutions


https://github.com/arduino/ArduinoTensorFlowLiteTutorials/

## Setup Python Environment

The next cell sets up the dependencies in required for the notebook, run it.

In [4]:
# Setup environment
!apt-get -qq install xxd
!pip install pandas numpy matplotlib
!pip install tensorflow

# Upload Data

1. Open the panel on the left side of Colab by clicking on the __>__
1. Select the files tab
1. Drag your gesture `.csv` files from your computer to the tab to upload them into colab. Use __one file per gesture__ and as many files as you like (e.g. `punch.csv`, `flex.csv`, `ML_Wave_Left.csv`, ...). The file name, without `.csv`, is used as the name of the gesture.

Each file can hold many recordings of the gesture, with a blank line between recordings. A complete recording is 119 rows of `aX,aY,aZ,gX,gY,gZ`. A header row, quotes around each row (what Excel produces) and Windows line endings are all fine. Incomplete or garbled recordings (for example the partial ones at the start and end of a capture) are skipped and reported.

In [ ]:
import glob
import os

import numpy as np

# one recording is one gesture performed once: 119 samples of aX, aY, aZ, gX, gY, gZ
SAMPLES_PER_GESTURE = 119
NUM_FEATURES = 6

# the range of the sensor, used to check the data and to normalize it later:
# - acceleration is between: -4 to +4
# - gyroscope is between: -2000 to +2000
SENSOR_RANGE = np.array([4, 4, 4, 2000, 2000, 2000])

# every .csv in /content is one gesture, named after the file
CSV_FILES = sorted(glob.glob("/content/*.csv"))
if not CSV_FILES:
  raise FileNotFoundError("No .csv files found in /content. Upload one csv per gesture first.")
print("Found", len(CSV_FILES), "csv files:", [os.path.basename(f) for f in CSV_FILES])


def load_recordings(path):
  """Return the complete recordings in a gesture csv as an array of shape
  (recordings, SAMPLES_PER_GESTURE, NUM_FEATURES).

  Recordings are separated by blank lines. Anything that is not exactly
  SAMPLES_PER_GESTURE rows of NUM_FEATURES readable numbers inside the range of
  the sensor is skipped, and the reason is printed.
  """
  # utf-8-sig drops the byte order mark that Excel adds
  with open(path, encoding="utf-8-sig") as f:
    file_lines = f.read().splitlines()

  # group the rows into recordings
  blocks, block = [], []
  for line in file_lines:
    row = line.strip().strip('"')  # Excel wraps each row in quotes
    if row[:1].isalpha():
      continue  # header row
    if row.strip(", "):
      block.append(row)
    elif block:  # a blank line ends the recording
      blocks.append(block)
      block = []
  if block:
    blocks.append(block)

  recordings, skipped = [], []
  for block in blocks:
    if len(block) != SAMPLES_PER_GESTURE:  # e.g. a capture that started or ended mid-gesture
      skipped.append(f"{len(block)} rows")
      continue
    try:
      values = np.array([[float(v) for v in row.split(",")] for row in block])
    except ValueError:  # not a number, or a row with a different number of values
      skipped.append("unreadable row")
      continue
    if values.shape != (SAMPLES_PER_GESTURE, NUM_FEATURES):
      skipped.append("unreadable row")
    elif (np.abs(values) > SENSOR_RANGE).any():  # e.g. a lost decimal point: 144.480 -> 144480
      skipped.append("value outside sensor range")
    else:
      recordings.append(values)

  summary = f"{os.path.basename(path)}: kept {len(recordings)} recordings"
  if skipped:
    summary += f", skipped {len(skipped)} ({', '.join(skipped)})"
  print(summary)
  return np.array(recordings).reshape(-1, SAMPLES_PER_GESTURE, NUM_FEATURES)

# Graph Data (optional)

We'll graph the input files on two separate graphs, acceleration and gyroscope, as each data set has different units and scale.

In [ ]:
import matplotlib.pyplot as plt

# the gesture file to graph, any of CSV_FILES
filename = CSV_FILES[0]

# stitch the complete recordings back together into one continuous trace
samples = load_recordings(filename).reshape(-1, NUM_FEATURES)

index = range(1, len(samples) + 1)

plt.rcParams["figure.figsize"] = (20,10)

plt.plot(index, samples[:, 0], 'g.', label='x', linestyle='solid', marker=',')
plt.plot(index, samples[:, 1], 'b.', label='y', linestyle='solid', marker=',')
plt.plot(index, samples[:, 2], 'r.', label='z', linestyle='solid', marker=',')
plt.title("Acceleration - " + os.path.basename(filename))
plt.xlabel("Sample #")
plt.ylabel("Acceleration (G)")
plt.legend()
plt.show()

plt.plot(index, samples[:, 3], 'g.', label='x', linestyle='solid', marker=',')
plt.plot(index, samples[:, 4], 'b.', label='y', linestyle='solid', marker=',')
plt.plot(index, samples[:, 5], 'r.', label='z', linestyle='solid', marker=',')
plt.title("Gyroscope - " + os.path.basename(filename))
plt.xlabel("Sample #")
plt.ylabel("Gyroscope (deg/sec)")
plt.legend()
plt.show()

# Train Neural Network





## Parse and prepare the data

The next cell parses the csv files and transforms them to a format that will be used to train the fully connected neural network.

Every csv file you uploaded becomes one gesture, so there is nothing to edit here. To add or remove a gesture, add or remove its file in the files tab and run the cells again.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

print(f"TensorFlow version = {tf.__version__}\n")

# Set a fixed random seed value, for reproducibility, this will allow us to get
# the same random numbers each time the notebook is run
SEED = 1337
np.random.seed(SEED)
tf.random.set_seed(SEED)

# the list of gestures that data is available for, named after the csv files
GESTURES = [os.path.splitext(os.path.basename(f))[0] for f in CSV_FILES]

NUM_GESTURES = len(GESTURES)

# create a one-hot encoded matrix that is used in the output
ONE_HOT_ENCODED_GESTURES = np.eye(NUM_GESTURES)

inputs = []
outputs = []

# which gesture each recording was first seen in, to catch the same recording
# showing up under two different gestures
first_seen_in = {}

# read each csv file and push an input and output
for gesture_index in range(NUM_GESTURES):
  gesture = GESTURES[gesture_index]
  print(f"Processing index {gesture_index} for gesture '{gesture}'.")

  output = ONE_HOT_ENCODED_GESTURES[gesture_index]

  recordings = load_recordings(CSV_FILES[gesture_index])

  print(f"\tThere are {len(recordings)} recordings of the {gesture} gesture.")

  copied_from = set()
  for recording in recordings:
    owner = first_seen_in.setdefault(recording.tobytes(), gesture)
    if owner != gesture:
      copied_from.add(owner)

    # normalize the input data, between 0 to 1, and flatten each recording into one input:
    # aX, aY, aZ, gX, gY, gZ for sample 1, then sample 2, ...
    inputs.append(((recording + SENSOR_RANGE) / (2 * SENSOR_RANGE)).flatten())
    outputs.append(output)

  if copied_from:
    print(f"\tWARNING: some of these recordings are identical to ones in {sorted(copied_from)}. "
          "The same data under two gestures can't be told apart, check the files aren't copies of each other.")

if len(inputs) == 0:
  raise ValueError("No complete recordings found in any csv, nothing to train on.")

# convert the list to numpy array
inputs = np.array(inputs)
outputs = np.array(outputs)

print("Data set parsing and preparation complete.")

## Randomize and split the input and output pairs for training

Randomly split input and output pairs into sets of data: 60% for training, 20% for validation, and 20% for testing.

  - the training set is used to train the model
  - the validation set is used to measure how well the model is performing during training
  - the testing set is used to test the model after training

In [ ]:
# Randomize the order of the inputs, so they can be evenly distributed for training, testing, and validation
# https://stackoverflow.com/a/37710486/2020087
num_inputs = len(inputs)
randomize = np.arange(num_inputs)
np.random.shuffle(randomize)

# Swap the consecutive indexes (0, 1, 2, etc) with the randomized indexes
inputs = inputs[randomize]
outputs = outputs[randomize]

# Split the recordings (group of samples) into three sets: training, testing and validation
TRAIN_SPLIT = int(0.6 * num_inputs)
TEST_SPLIT = int(0.2 * num_inputs + TRAIN_SPLIT)

inputs_train, inputs_test, inputs_validate = np.split(inputs, [TRAIN_SPLIT, TEST_SPLIT])
outputs_train, outputs_test, outputs_validate = np.split(outputs, [TRAIN_SPLIT, TEST_SPLIT])

print("Data set randomization and splitting complete.")

## Build & Train the Model

Build and train a [TensorFlow](https://www.tensorflow.org) model using the high-level [Keras](https://www.tensorflow.org/guide/keras) API.

In [ ]:
# build the model and train it
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(50, activation='relu')) # relu is used for performance
model.add(tf.keras.layers.Dense(15, activation='relu'))
model.add(tf.keras.layers.Dense(NUM_GESTURES, activation='softmax')) # softmax is used, because we only expect one gesture to occur per input
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history = model.fit(inputs_train, outputs_train, epochs=600, batch_size=1, validation_data=(inputs_validate, outputs_validate))



## Verify

Graph the models performance vs validation.


### Graph the loss

Graph the loss to see when the model stops improving.

In [ ]:
# increase the size of the graphs. The default size is (6,4).
plt.rcParams["figure.figsize"] = (20,10)

# graph the loss, the model above is configure to use "mean squared error" as the loss function
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'g.', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

print(plt.rcParams["figure.figsize"])

### Graph the loss again, skipping a bit of the start

We'll graph the same data as the previous code cell, but start at index 100 so we can further zoom in once the model starts to converge.

In [ ]:
# graph the loss again skipping a bit of the start
SKIP = 100
plt.plot(epochs[SKIP:], loss[SKIP:], 'g.', label='Training loss')
plt.plot(epochs[SKIP:], val_loss[SKIP:], 'b.', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Graph the mean absolute error

[Mean absolute error](https://en.wikipedia.org/wiki/Mean_absolute_error) is another metric to judge the performance of the model.



In [ ]:
# graph of mean absolute error
mae = history.history['mae']
val_mae = history.history['val_mae']
plt.plot(epochs[SKIP:], mae[SKIP:], 'g.', label='Training MAE')
plt.plot(epochs[SKIP:], val_mae[SKIP:], 'b.', label='Validation MAE')
plt.title('Training and validation mean absolute error')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.show()


### Run with Test Data
Put our test data into the model and plot the predictions


In [ ]:
# use the model to predict the test inputs
predictions = model.predict(inputs_test)

# print the predictions and the expected ouputs
print("predictions =\n", np.round(predictions, decimals=3))
print("actual =\n", outputs_test)

# the most likely gesture for each test recording, compared to the real one
predicted_gesture = np.argmax(predictions, axis=1)
actual_gesture = np.argmax(outputs_test, axis=1)

# count how often each real gesture (row) was predicted as each gesture (column)
confusion = np.zeros((NUM_GESTURES, NUM_GESTURES), dtype=int)
for actual, predicted in zip(actual_gesture, predicted_gesture):
  confusion[actual, predicted] += 1

print(f"\n{np.sum(predicted_gesture == actual_gesture)} of {len(actual_gesture)} test recordings correct")
for i in range(NUM_GESTURES):
  print(f"\t{GESTURES[i]}: {confusion[i, i]} of {confusion[i].sum()}")

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(confusion, cmap='Blues')
for i in range(NUM_GESTURES):
  for j in range(NUM_GESTURES):
    plt.text(j, i, confusion[i, j], ha='center', va='center',
             color='white' if confusion[i, j] > confusion.max() / 2 else 'black')
plt.xticks(range(NUM_GESTURES), GESTURES, rotation=45, ha='right')
plt.yticks(range(NUM_GESTURES), GESTURES)
plt.xlabel('Predicted gesture')
plt.ylabel('Actual gesture')
plt.title('Test data predicted vs actual gestures')
plt.tight_layout()
plt.show()

# Convert the Trained Model to Tensor Flow Lite

The next cell converts the model to TFlite format. The size in bytes of the model is also printed out.

In [ ]:
# Convert the model to the TensorFlow Lite format without quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model to disk
open("gesture_model.tflite", "wb").write(tflite_model)

import os
basic_model_size = os.path.getsize("gesture_model.tflite")
print("Model is %d bytes" % basic_model_size)



## Encode the Model in an Arduino Header File

The next cell creates a constant byte array that contains the TFlite model. Import it as a tab with the sketch below.

In [ ]:
!echo "const unsigned char model[] = {" > /content/model.h
!cat gesture_model.tflite | xxd -i      >> /content/model.h
!echo "};"                              >> /content/model.h

import os
model_h_size = os.path.getsize("model.h")
print(f"Header file, model.h, is {model_h_size:,} bytes.")
print("\nOpen the side panel (refresh if needed). Double click model.h to download the file.")

# Classifying IMU Data

Now it's time to switch back to the tutorial instructions and run our new model on the Arduino Nano 33 BLE Sense to classify the accelerometer and gyroscope data.
